# 🏛️ UNIVERSIDAD SANTO TOMÁS · SECCIONAL TUNJA
## División de Ingenierías y Arquitectura — Facultad de Ingeniería de Sistemas
### Espacio Académico: **Big Data (2026-2)**
---
# 📊 PROYECTO INTEGRADOR: ARQUITECTURA LAKEHOUSE Y ANALÍTICA DISTRIBUIDA DE PRECIOS DE MODELOS DE INTELIGENCIA ARTIFICIAL A GRAN ESCALA (2023–2026)

**Integrantes del Equipo:**
1. **Juan Esteban Silva Espejo**
2. **Michael Arles Gómez Bernal**

**Repositorio del Proyecto (GitHub):** [https://github.com/JuanEstebanSilva/BigData.git](https://github.com/JuanEstebanSilva/BigData.git)  
**Dataset Oficial:** *AI Model Price History (2023–2026)* — 167.787 registros (~16,7 MB)  
**Fecha de Entrega:** Noviembre de 2026  
**Modalidad de Ejecución:** Google Colab / PySpark Distributed Environment

---
### 📌 Síntesis Ejecutiva del Proyecto
El presente proyecto integrador diseña e implementa un flujo integral de **Big Data (End-to-End Data Pipeline)** para el análisis masivo de la evolución temporal y la dispersión económica en el mercado global de APIs de **Modelos Fundacionales de Inteligencia Artificial (LLMs y Embeddings)** entre septiembre de 2023 y septiembre de 2026.

El flujo abarca:
1. **Ingesta y Lakehouse con Delta Lake (Electivas B y C):** Arquitectura Medallón en capas (*Landing* → *Bronze* → *Silver* → *Gold*), garantizando propiedades **ACID**, versionado histórico (**Time Travel**) y control estricto de esquemas (**Schema Enforcement & Evolution**).
2. **Limpieza y Optimización en Memoria (Obligatoria 2 & Unidad 5):** Reducción de más del 70 % en la huella de memoria RAM mediante tipificación eficiente, imputación justificada y transformaciones vectorizadas con NumPy.
3. **Procesamiento Distribuido con Apache Spark y Spark SQL (Obligatoria 1 & Unidad 3):** Procesamiento a gran escala con PySpark DataFrames, transformaciones, agregaciones complejas con `groupBy()` y consultas analíticas con funciones de ventana en Spark SQL, verificadas con `.count()`.
4. **Estadística Descriptiva a Gran Escala (Unidad 6):** Tendencia central robusta, dispersión, asimetría, curtosis, percentiles de cola (p1 a p99.9), detección de atípicos con la regla de Tukey ($Q_3 + 1.5 \cdot IQR$) y cálculo exacto vs. aproximado con DuckDB (T-Digest).
5. **Visualización y Prototipo Analítico (Obligatoria 2):** Gráficos exploratorios en Seaborn/Matplotlib y visualizaciones interactivas en Plotly preparadas para el consumo gerencial y soporte del futuro dashboard.
6. **Simulación de Ingesta en Streaming con PySpark Structured Streaming (Electiva D):** Micro-batches en tiempo real de actualizaciones de precios simulando una fuente viva tipo Kafka.


## 🛠️ Paso 0 · Instalación de Librerías y Configuración del Entorno (≈ 1 minuto)
Para ejecutar este flujo completo en Google Colab o entorno local, instalamos las librerías vistas en el syllabus:
- `pyspark`: Motor de procesamiento distribuido para Big Data y Spark SQL.
- `deltalake`: Motor de almacenamiento transaccional Lakehouse (soporta transacciones ACID, Time Travel y esquema).
- `duckdb`: Motor analítico columnar OLAP embebido en memoria de altísima velocidad.
- `polars`: Procesamiento distribuido y multihilo en memoria para estadística a gran escala.
- `pandas`, `numpy`, `matplotlib`, `seaborn`, `plotly`: Analítica, vectorización y visualizaciones interactivas.


In [ ]:
# Instalación silenciosa de dependencias en el entorno de ejecución
%pip install -q pyspark deltalake duckdb polars pandas numpy matplotlib seaborn plotly


### Importación de módulos y configuración de visualización


In [ ]:
import os
import sys
import time
import json
import math
import shutil
from datetime import datetime

# Computación científica y analítica
import numpy as np
import pandas as pd
import polars as pl
import duckdb

# Visualización gráfica
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Lakehouse y Delta Lake
from deltalake import DeltaTable, write_deltalake

# Procesamiento Distribuido PySpark
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, DateType
)
from pyspark.sql.window import Window

# Configuración de estilos visuales
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print(f"✅ Entorno listo.")
print(f"   - Python: {sys.version.split()[0]}")
print(f"   - PySpark: {pyspark.__version__}")
print(f"   - Pandas: {pd.__version__}")
print(f"   - Polars: {pl.__version__}")
print(f"   - DuckDB: {duckdb.__version__}")


---
# 🎯 1. Definición del Contexto y Preguntas de Análisis (Tarea T1)

### 1.1 Sector y Problema de Negocio
El sector tecnológico atraviesa una transformación histórica impulsada por la **Inteligencia Artificial Generativa**. La comercialización masiva de Modelos de Lenguaje Grande (LLMs) y de Embeddings se realiza predominantemente bajo el modelo de **Inferencia como Servicio (IaaS / MaaS)** mediante APIs en la nube. En este modelo, los proveedores facturan a los desarrolladores y corporaciones con base en el consumo de **tokens** procesados (divididos en tokens de entrada o *prompt* y tokens de salida o *completion*).

Sin embargo, el ecosistema presenta tres desafíos económicos y operacionales críticos:
1. **Volatilidad y Guerra de Precios:** Entre finales de 2023 y finales de 2026, la competencia entre desarrolladores líderes (**OpenAI, Meta, Anthropic, Google, Mistral, DeepSeek**) ha provocado una reducción acelerada en los costos de cómputo por token (la denominada *"deflación de la inferencia"*).
2. **Disparidad por Proveedor de Nube (*Cloud Arbitrage*):** Un mismo modelo de código abierto (por ejemplo, la familia **Meta Llama 3**) es ofrecido simultáneamente por múltiples plataformas (AWS Bedrock, Microsoft Azure AI, Fireworks AI, Groq, DeepInfra, OpenRouter) con variaciones de precio que superan el 300 % y diferentes políticas de cobro según la latencia y la tasa de rendimiento.
3. **Desacoplamiento entre Ventana de Contexto y Costo:** La expansión de ventanas de contexto desde 4.000 tokens hasta más de 1.000.000 y 10.000.000 de tokens ha transformado el diseño de aplicaciones, pero genera incertidumbre en los *Unit Economics* corporativos.

### 1.2 Usuarios de la Información
- **Directores de Tecnología (CTO) y VPs de Ingeniería:** Para seleccionar el mix óptimo de proveedores y modelos minimizando el costo operativo mensual (*OpEx*).
- **Ingenieros de MLOps y Arquitectos de Software:** Para diseñar sistemas de enrutamiento inteligente de consultas (*LLM Routers* o *Semantic Caching*) dirigiendo solicitudes a proveedores más económicos según la longitud del prompt.
- **CFOs y Equipos de FinOps Cloud:** Para auditar la variabilidad de costos, negociar contratos corporativos con nubes públicas y proyectar el ROI de proyectos basados en IA.

### 1.3 Formulación de las 5 Preguntas de Investigación Concretas y Medibles
Las siguientes preguntas guiarán todo el desarrollo analítico y técnico del proyecto:

| # | Pregunta de Análisis | Métrica / Indicador | Meta Cuantitativa |
|---|---|---|---|
| **P1** | ¿Existe una tendencia deflacionaria medible en los precios de inferencia por millón de tokens entre 2023 y 2026 para los principales desarrolladores (OpenAI, Meta, Anthropic, Google)? | Tasa de variación porcentual interanual de precios de entrada y salida ($USD/1M tokens) | Cuantificar la caída porcentual media y por desarrollador entre 2023 y 2026. |
| **P2** | ¿Qué nivel de disparidad de precios existe entre proveedores de nube para una misma familia de modelos de código abierto (ej. Meta Llama 3)? | Rango, desviación estándar y ratio $\frac{\text{Precio Máximo}}{\text{Precio Mínimo}}$ entre proveedores para la misma familia | Identificar el proveedor más económico y cuantificar el sobrecosto porcentual del proveedor más caro. |
| **P3** | ¿Existe correlación estadística entre la capacidad de la ventana de contexto (`max_input_tokens`) y las tarifas cobradas por millón de tokens? | Coeficientes de correlación de Pearson y Spearman entre ventana de contexto y precios | Comprobar si ventanas ultra-largas ($>1M$ tokens) penalizan el precio unitario del token o si el costo es independiente. |
| **P4** | ¿Cuál es la estructura de costos según la modalidad del modelo (`chat`, `completion`, `embedding`, `responses`) y cuál es la brecha media entre tokens de entrada y salida? | Mediana y media de precio por modalidad; ratio medio $\frac{\text{Precio Salida}}{\text{Precio Entrada}}$ | Determinar el multiplicador exacto que cuesta generar texto frente a procesar texto. |
| **P5** | ¿Cuáles son los modelos y proveedores más costo-eficientes para una carga de trabajo corporativa típica (1.000 tokens entrada / 500 tokens salida) y cómo se caracterizan los valores atípicos extremos? | Costo estimado por consulta típica ($USD) y umbral de Tukey ($Q_3 + 1.5 \cdot IQR$) | Rankear el top 10 más económico y cuantificar el porcentaje de registros que constituyen atípicos de alto costo. |


---
# 📁 2. Selección y Descripción del Dataset (Tarea T2)

### 2.1 Ficha Técnica del Conjunto de Datos
- **Nombre:** *AI Model Price History (2023–2026)*
- **Fuente de Origen:** Seguimiento semanal agregado de tarifas públicas de inferencia en APIs compilado a partir de Artificial Analysis y OpenRouter API Catalog.
- **Repositorio Oficial:** [GitHub JuanEstebanSilva/BigData](https://github.com/JuanEstebanSilva/BigData.git)
- **Licencia:** *Open Data Commons Attribution License (ODC-By / Creative Commons Attribution 4.0 International)*. Permite uso académico y comercial citando la fuente.
- **Volumen Total:** **167.787 registros** y **9 columnas**, con un peso en disco de **16,70 MB**. Cumple holgadamente con el requisito obligatorio de Big Data (mínimo 100.000 registros).
- **Cobertura Temporal:** 157 semanas continuas, desde el **18 de septiembre de 2023** hasta el **28 de septiembre de 2026**.

### 2.2 Diccionario de Datos Formal
| # | Columna | Tipo de Dato Físico | Tipo Semántico | Descripción Detallada | Rango / Valores Ejemplo |
|---|---|---|---|---|---|
| 0 | `week_start` | `object` → `date` | Temporal | Fecha de inicio de la semana de vigencia de la tarifa (lunes). Formato ISO 8601 `AAAA-MM-DD`. | `2023-09-18` a `2026-09-28` |
| 1 | `model_id` | `object` | Identificador | Identificador único de endpoint del modelo ante el proveedor o API gateway. | `openai/gpt-4o`, `meta/llama-3-70b-instruct` |
| 2 | `model_name` | `object` | Categórico nominal | Nombre comercial legible del modelo de inteligencia artificial. | `gpt-4o-mini`, `llama-v3p2-90b-vision` |
| 3 | `developer` | `object` | Categórico nominal | Empresa u organización que diseñó y entrenó la arquitectura del modelo. | `OpenAI`, `Meta`, `Anthropic`, `Google`, `Mistral AI` |
| 4 | `provider` | `object` | Categórico nominal | Infraestructura o plataforma en la nube que hospeda el modelo y factura la API. | `bedrock`, `azure_ai`, `fireworks_ai`, `groq`, `openrouter` |
| 5 | `mode` | `object` | Categórico ordinal | Tipo o modalidad funcional de inferencia del modelo. | `chat`, `embedding`, `completion`, `responses` |
| 6 | `input_price_per_1m_tokens` | `float64` | Numérico continuo | Tarifa en dólares estadounidenses (USD) por procesar 1 millón de tokens de entrada (prompt). | $0.00$ a $250.00$ USD |
| 7 | `output_price_per_1m_tokens` | `float64` | Numérico continuo | Tarifa en dólares estadounidenses (USD) por generar 1 millón de tokens de salida (completion). | $0.00$ a $1000.00$ USD |
| 8 | `max_input_tokens` | `float64` → `int32` | Numérico discreto | Capacidad máxima de la ventana de contexto en número de tokens de entrada simultáneos. | $77$ a $10.485.760$ tokens |

### 2.3 Cumplimiento de Normativa de Protección de Datos (Ley 1581 de 2012)
> [!NOTE]
> **Declaración de Conformidad con la Ley 1581 de 2012 (Habeas Data):**  
> El conjunto de datos analizado contiene exclusivamente metadatos comerciales, técnicos y tarifarios de software y servicios de computación en la nube disponibles públicamente en internet. **NO contiene datos personales, datos biométricos, información crediticia, ni ningún dato sensible relativo a personas naturales identificadas o identificables**. Por ende, no requiere procesos de anonimización ni consentimientos individuales previos, garantizando el pleno cumplimiento de la legislación colombiana e internacional en gobernanza de datos.


---
# 🏗️ 3. Arquitectura del Flujo de Datos y Lakehouse Medallion (Tareas T3, Electivas B y C)

El proyecto adopta el paradigma moderno de **Lakehouse** basado en la **Arquitectura Medallón** (*Medallion Architecture*), combinando la escalabilidad y bajo costo de un Data Lake de objetos con las garantías de confiabilidad transaccional ACID de un Data Warehouse corporativo:

```mermaid
flowchart LR
    A["🌐 Fuentes Externas
(APIs de Proveedores:
OpenAI, Bedrock, Fireworks)"] --> B["📥 Capa LANDING
(Archivo CSV Crudo:
ai_model_price_history.csv)"]
    B --> C["🥉 Capa BRONZE
(Delta Lake Crudo:
Parquet + _delta_log
Preservación histórica)"]
    C --> D["🥈 Capa SILVER
(Delta Lake Limpio:
Tipos normalizados,
ACID Merge, Schema Control)"]
    D --> E["🥇 Capa GOLD
(Tablas Analíticas Agregadas:
KPIs, DuckDB & Spark SQL)"]
    E --> F["📊 Consumo & Analítica
(PySpark Distributed,
Estadística U6,
Gráficos Plotly & BI)"]
```

### Descripción de las Capas y Tecnologías
1. **Landing Zone:** Ingesta del archivo fuente CSV crudo tal como proviene de la fuente externa, garantizando reproducibilidad total.
2. **Capa Bronze (Raw Delta):** Conversión directa a formato **Delta Lake**. Preserva los datos crudos en archivos columnares **Apache Parquet**, respaldados por el registro secuencial de transacciones **`_delta_log`** (archivos JSON).
3. **Capa Silver (Cleansed Delta):** Datos sometidos a validación de reglas de calidad: eliminación de valores inválidos, estandarización de tipos numéricos y fechas, adición de columnas calculadas vectorizadas y aplicación de transacciones **MERGE (Upsert)** bajo control de esquema (**Schema Enforcement y Schema Evolution**).
4. **Capa Gold (Curated Business Marts):** Modelos agregados y vistas analíticas preparadas para consultas de alto rendimiento con **DuckDB** y **Spark SQL**, reduciendo el volumen de escaneo y optimizando tiempos de respuesta para visualización y dashboards.


---
# 🔍 4. Diagnóstico de Calidad Inicial (Tareas T4 y T5)

Antes de cualquier procesamiento, se realiza una inspección automatizada de calidad sobre el dataset crudo, alineada con las dimensiones del estándar **ISO/IEC 5259** (Completitud, Exactitud, Consistencia, Unicidad y Validez).


In [ ]:
# Ubicación del archivo en el entorno local o Google Colab
CSV_PATH = "ai_model_price_history.csv"

# Si no existe en el directorio actual, verificar en la ruta raíz del repositorio
if not os.path.exists(CSV_PATH):
    if os.path.exists("../ai_model_price_history.csv"):
        CSV_PATH = "../ai_model_price_history.csv"
    else:
        # Descarga automática de respaldo desde el repositorio GitHub si se ejecuta en Colab nuevo
        print("Descargando dataset desde el repositorio oficial...")
        !wget -q https://raw.githubusercontent.com/JuanEstebanSilva/BigData/main/ai_model_price_history.csv -O ai_model_price_history.csv

# Ingesta cruda con Pandas para diagnóstico inicial
df_crudo = pd.read_csv(CSV_PATH)
tamano_mb = os.path.getsize(CSV_PATH) / (1024 * 1024)

print(f"📊 Dataset Crudo Cargado:")
print(f"   - Total de Filas: {len(df_crudo):,}")
print(f"   - Total de Columnas: {len(df_crudo.columns)}")
print(f"   - Tamaño en Disco: {tamano_mb:.2f} MB")
print(f"   - Uso en Memoria RAM: {df_crudo.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
df_crudo.head(3)


In [ ]:
# Diagnóstico Exhaustivo de Calidad por Columna (ISO/IEC 5259)
diagnostico = []

for col in df_crudo.columns:
    nulos = df_crudo[col].isna().sum()
    pct_nulos = (nulos / len(df_crudo)) * 100
    unicos = df_crudo[col].nunique()
    tipo = str(df_crudo[col].dtype)
    ejemplo = str(df_crudo[col].dropna().iloc[0]) if unicos > 0 else "N/A"
    
    diagnostico.append({
        "Columna": col,
        "Tipo Físico": tipo,
        "Valores No Nulos": len(df_crudo) - nulos,
        "Nulos": nulos,
        "% Nulos": round(pct_nulos, 3),
        "Valores Únicos": unicos,
        "Ejemplo": ejemplo[:40]
    })

df_diag = pd.DataFrame(diagnostico)
duplicados_totales = df_crudo.duplicated().sum()

print(f"🔎 Duplicados exactos en el dataset: {duplicados_totales}")
print(f"🔎 Rango temporal: {df_crudo['week_start'].min()} a {df_crudo['week_start'].max()} ({df_crudo['week_start'].nunique()} semanas)")
df_diag


### 🧠 Qué observar en el Diagnóstico de Calidad:
1. **Completitud y Ausencia de Duplicados:** Existen **0 filas duplicadas exactas**, confirmando la alta granularidad de la fuente semanal.
2. **Nulos en `output_price_per_1m_tokens` (1.144 nulos, 0,68 %):** No constituyen un error de captura; la gran mayoría corresponden a modelos con modalidad `embedding`. Los modelos de embeddings convierten texto en vectores densos y **no generan texto de salida**, por lo que su tarifa de salida es por definición nula o cero.
3. **Nulos en `input_price_per_1m_tokens` (1.045 nulos, 0,62 %):** Modelos en fase de vista previa privada (*preview*) o en proceso de despliegue donde los proveedores no habían publicado la tarifa oficial al momento del registro.
4. **Nulos en `max_input_tokens` (13.361 nulos, 7,96 %):** Registros históricos de modelos antiguos o arquitecturas experimentales donde la longitud de ventana no fue declarada en el catálogo.
5. **Nulos en `model_name` (42 filas, 0,025 %):** Ocurren únicamente en endpoints de Fireworks AI que terminaban en barra `/` sin especificar el identificador del modelo final.


---
# 🧹 5. Limpieza, Optimización de Memoria y Transformación Vectorizada (Tarea T5)

En Big Data, optimizar la memoria en los nodos de cómputo es crítico. Convertiremos tipos de texto genéricos (`object`) a categorías (`category`), fechas nativas (`datetime64[ns]`), enteros reducidos (`int32`) y flotantes optimizados (`float32`), aplicando reglas de depuración con justificación técnica formal.


In [ ]:
# Medición de memoria previa
mem_antes_mb = df_crudo.memory_usage(deep=True).sum() / (1024**2)

# Copia de trabajo para el pipeline de limpieza
df_limpio = df_crudo.copy()

# 1. Regla: Depurar 42 filas sin model_name (endpoints incompletos)
df_limpio = df_limpio.dropna(subset=["model_name"]).copy()

# 2. Regla: Conversión estricta de fecha
df_limpio["week_start"] = pd.to_datetime(df_limpio["week_start"])

# 3. Regla: Modelos de embedding tienen tarifa de salida $0.00
df_limpio.loc[(df_limpio["mode"] == "embedding") & (df_limpio["output_price_per_1m_tokens"].isna()), "output_price_per_1m_tokens"] = 0.0

# 4. Regla: Filtrar registros con precios de entrada nulos (1.021 registros sin precio comercial)
df_limpio = df_limpio.dropna(subset=["input_price_per_1m_tokens", "output_price_per_1m_tokens"]).copy()

# 5. Regla: Imputar ventana de contexto faltante con la mediana por desarrollador (o mediana global)
mediana_contexto = df_limpio.groupby("developer")["max_input_tokens"].transform("median")
df_limpio["max_input_tokens"] = df_limpio["max_input_tokens"].fillna(mediana_contexto).fillna(df_limpio["max_input_tokens"].median()).astype("int32")

# 6. Regla: Optimización de tipos (Downcasting)
df_limpio["developer"] = df_limpio["developer"].astype("category")
df_limpio["provider"] = df_limpio["provider"].astype("category")
df_limpio["mode"] = df_limpio["mode"].astype("category")
df_limpio["input_price_per_1m_tokens"] = df_limpio["input_price_per_1m_tokens"].astype("float32")
df_limpio["output_price_per_1m_tokens"] = df_limpio["output_price_per_1m_tokens"].astype("float32")

# ========================================================
# TRANSFORMACIÓN VECTORIZADA CON NUMPY (Nuevas Variables)
# ========================================================
# A) Precio total equilibrado (1M entrada + 1M salida)
df_limpio["total_price_1m"] = df_limpio["input_price_per_1m_tokens"] + df_limpio["output_price_per_1m_tokens"]

# B) Ratio salida / entrada (cuántas veces es más cara la generación que la lectura)
df_limpio["price_ratio_output_input"] = np.where(
    df_limpio["input_price_per_1m_tokens"] > 0,
    np.round(df_limpio["output_price_per_1m_tokens"] / df_limpio["input_price_per_1m_tokens"], 3),
    0.0
).astype("float32")

# C) Costo en USD de una consulta empresarial típica: 1.000 tokens de prompt + 500 tokens de respuesta
# Fórmula: (input * 1.000 + output * 500) / 1.000.000
df_limpio["cost_query_typical"] = (
    (df_limpio["input_price_per_1m_tokens"] * 1000.0 + df_limpio["output_price_per_1m_tokens"] * 500.0) / 1_000_000.0
).astype("float32")

# D) Transformación logarítmica para atenuar asimetría extrema (para correlación y ML)
df_limpio["log_input_price"] = np.log1p(df_limpio["input_price_per_1m_tokens"]).astype("float32")
df_limpio["log_output_price"] = np.log1p(df_limpio["output_price_per_1m_tokens"]).astype("float32")

# E) Segmentación categórica de tamaño de ventana de contexto
condiciones_contexto = [
    df_limpio["max_input_tokens"] < 32768,
    (df_limpio["max_input_tokens"] >= 32768) & (df_limpio["max_input_tokens"] < 131072),
    (df_limpio["max_input_tokens"] >= 131072) & (df_limpio["max_input_tokens"] < 1048576),
    df_limpio["max_input_tokens"] >= 1048576
]
etiquetas_contexto = ["1. Estándar (<32k)", "2. Mediana (32k-128k)", "3. Grande (128k-1M)", "4. Masiva (>=1M)"]
df_limpio["context_category"] = pd.Categorical(
    np.select(condiciones_contexto, etiquetas_contexto, default="1. Estándar (<32k)")
)

# Medición final de memoria
mem_despues_mb = df_limpio.memory_usage(deep=True).sum() / (1024**2)
reduccion_pct = ((mem_antes_mb - mem_despues_mb) / mem_antes_mb) * 100

print(f"🚀 Limpieza y Transformación Vectorizada Concluida:")
print(f"   - Memoria ANTES:   {mem_antes_mb:.2f} MB")
print(f"   - Memoria DESPUÉS: {mem_despues_mb:.2f} MB")
print(f"   - Ahorro de Memoria: {reduccion_pct:.2f} %")
print(f"   - Filas finales:   {len(df_limpio):,} (se retuvieron {len(df_limpio)/len(df_crudo)*100:.2f} % de datos válidos)")


In [ ]:
# Tabla Resumen de Métricas de Calidad: ANTES vs. DESPUÉS (Numeral 5 del PDF)
metricas_comparativas = pd.DataFrame({
    "Dimensión / Métrica": [
        "Total de Filas",
        "Total de Columnas",
        "Nulos en model_name",
        "Nulos en input_price",
        "Nulos en output_price",
        "Nulos en max_input_tokens",
        "Uso de Memoria RAM (MB)",
        "Tipos de Datos Eficientes"
    ],
    "Estado ANTES (Crudo)": [
        f"{len(df_crudo):,}",
        str(len(df_crudo.columns)),
        f"{df_crudo['model_name'].isna().sum():,}",
        f"{df_crudo['input_price_per_1m_tokens'].isna().sum():,}",
        f"{df_crudo['output_price_per_1m_tokens'].isna().sum():,}",
        f"{df_crudo['max_input_tokens'].isna().sum():,}",
        f"{mem_antes_mb:.2f} MB",
        "object, float64 (Sin optimizar)"
    ],
    "Estado DESPUÉS (Limpio)": [
        f"{len(df_limpio):,}",
        str(len(df_limpio.columns)),
        "0 (Filtradas 42 sin nombre)",
        "0 (Imputados/Filtrados)",
        "0 (Corregidos embeddings a $0)",
        "0 (Imputados por mediana)",
        f"{mem_despues_mb:.2f} MB",
        "category, int32, float32, date"
    ],
    "Impacto / Decisión": [
        f"Depuración del {((len(df_crudo)-len(df_limpio))/len(df_crudo))*100:.2f} % no representativo",
        "+6 columnas analíticas vectorizadas",
        "Eliminación de URLs vacías en Fireworks",
        "Conservación exclusiva de tarifas reales",
        "Corrección semántica para embeddings",
        "Estimación estadística robusta",
        f"Reducción del {reduccion_pct:.1f} % en huella RAM",
        "Alineado con prácticas de alto rendimiento"
    ]
})
metricas_comparativas


---
# 🏛️ 6. Implementación Lakehouse con Delta Lake: Transacciones ACID, Time Travel y Control de Esquema (Electivas B y C)

Siguiendo la metodología vista en clase en el **`Lab_Lakehouse_Colab.ipynb`**, implementaremos de manera real las capas Medallón utilizando **Delta Lake**:
1. **Capa Bronze:** Conversión de los datos crudos a tabla Delta Lake.
2. **Inspección de `_delta_log`:** Demostración de las transacciones atómicas con Parquet + JSON log.
3. **Capa Silver:** Persistencia de la versión depurada.
4. **Operación MERGE (ACID Upsert):** Actualización de precios e inserción de nuevos modelos en una única transacción atómica.
5. **Time Travel:** Consulta auditable del estado de la tabla en versiones históricas previas.
6. **Control de Esquema:** Demostración de *Schema Enforcement* (rechazo de columna imprevista) y posterior *Schema Evolution* (`schema_mode="merge"`).
7. **Capa Gold con DuckDB:** Vistas analíticas de agregación consultadas a velocidad ultrarrápida.


In [ ]:
# Directorio local o del entorno Colab para el almacenamiento del Lakehouse
LAKEHOUSE_DIR = "lakehouse_storage"
if os.path.exists(LAKEHOUSE_DIR):
    shutil.rmtree(LAKEHOUSE_DIR)

os.makedirs(f"{LAKEHOUSE_DIR}/bronze", exist_ok=True)
os.makedirs(f"{LAKEHOUSE_DIR}/silver", exist_ok=True)
os.makedirs(f"{LAKEHOUSE_DIR}/gold", exist_ok=True)

# ----------------------------------------------------
# PASO 1: Persistir Capa BRONZE (Datos crudos en Delta)
# ----------------------------------------------------
bronze_uri = f"{LAKEHOUSE_DIR}/bronze/ai_prices"
write_deltalake(bronze_uri, df_crudo.astype(str), mode="overwrite", schema_mode="overwrite")

print(f"✅ Capa BRONZE creada exitosamente en '{bronze_uri}'")
archivos_bronze = os.listdir(bronze_uri)
print(f"   Archivos generados: {len(archivos_bronze)} (incluye Parquets y carpeta _delta_log)")


In [ ]:
# PASO 2: Por dentro de una tabla Delta: Inspección del diario de transacciones (_delta_log)
log_dir = f"{bronze_uri}/_delta_log"
log_files = sorted(os.listdir(log_dir))
print(f"Archivos en _delta_log: {log_files}")

# Leer el primer commit (Versión 0)
with open(f"{log_dir}/00000000000000000000.json", "r", encoding="utf-8") as f:
    lineas_commit_0 = f.readlines()

print(f"\nAcciones registradas en el commit 0 (primeras 5 acciones):")
for l in lineas_commit_0[:5]:
    accion = json.loads(l)
    tipo = list(accion.keys())[0]
    print(f" - Acción ACID: {tipo:12s} -> {str(accion[tipo])[:70]}...")


### 🧠 Qué observar en la Capa Bronze y el Diario `_delta_log`:
El archivo `00000000000000000000.json` certifica la transacción atómica de creación. Cada registro `add` asocia un archivo Parquet inmutable a la tabla. **Si un archivo Parquet llegara a corromperse o escribirse a medias, al no estar registrado en el commit del diario, para Delta Lake simplemente no existe.** Esto garantiza la propiedad de **Atomicidad y Consistencia (ACID)**.


In [ ]:
# ----------------------------------------------------
# PASO 3: Persistir Capa SILVER (Datos limpios y enriquecidos)
# ----------------------------------------------------
silver_uri = f"{LAKEHOUSE_DIR}/silver/ai_prices"
# Convertir columnas de categoría a string/tipos primitivos para compatibilidad Arrow/Delta
df_silver_export = df_limpio.copy()
for col in ["developer", "provider", "mode", "context_category"]:
    df_silver_export[col] = df_silver_export[col].astype(str)

write_deltalake(silver_uri, df_silver_export, mode="overwrite", schema_mode="overwrite")
dt_silver = DeltaTable(silver_uri)
print(f"✅ Capa SILVER creada exitosamente en versión {dt_silver.version()} con {len(df_silver_export):,} registros.")


In [ ]:
# ----------------------------------------------------
# PASO 4: Demostración de Transacción MERGE (ACID Upsert)
# ----------------------------------------------------
# Supongamos que llega un lote de ajuste del catálogo:
# 1) Actualización de precio para 'gpt-4o-mini' en OpenAI
# 2) Inserción de un modelo totalmente nuevo 'gpt-5-preview'

lote_actualizaciones = pd.DataFrame([
    {
        "week_start": pd.to_datetime("2026-10-05"),
        "model_id": "openai/gpt-4o-mini",
        "model_name": "gpt-4o-mini",
        "developer": "OpenAI",
        "provider": "openai",
        "mode": "chat",
        "input_price_per_1m_tokens": 0.10,   # Precio reducido de oferta
        "output_price_per_1m_tokens": 0.40,  # Precio reducido de oferta
        "max_input_tokens": 128000,
        "total_price_1m": 0.50,
        "price_ratio_output_input": 4.0,
        "cost_query_typical": 0.0003,
        "log_input_price": np.log1p(0.10),
        "log_output_price": np.log1p(0.40),
        "context_category": "2. Mediana (32k-128k)"
    },
    {
        "week_start": pd.to_datetime("2026-10-05"),
        "model_id": "openai/gpt-5-orion-preview",
        "model_name": "gpt-5-orion-preview",
        "developer": "OpenAI",
        "provider": "openai",
        "mode": "chat",
        "input_price_per_1m_tokens": 1.25,
        "output_price_per_1m_tokens": 5.00,
        "max_input_tokens": 2000000,
        "total_price_1m": 6.25,
        "price_ratio_output_input": 4.0,
        "cost_query_typical": 0.00375,
        "log_input_price": np.log1p(1.25),
        "log_output_price": np.log1p(5.00),
        "context_category": "4. Masiva (>=1M)"
    }
])

# Ejecución de MERGE ACID
resultado_merge = (
    dt_silver.merge(
        source=lote_actualizaciones,
        predicate="t.model_id = s.model_id AND t.week_start = s.week_start",
        source_alias="s",
        target_alias="t"
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

print(f"✅ Transacción MERGE completada exitosamente:")
print(f"   - Filas insertadas: {resultado_merge.get('num_target_rows_inserted', 0)}")
print(f"   - Filas actualizadas: {resultado_merge.get('num_target_rows_updated', 0)}")
print(f"   - Nueva Versión de la Tabla Delta: {DeltaTable(silver_uri).version()}")


In [ ]:
# ----------------------------------------------------
# PASO 5: Demostración de Time Travel (Viaje en el Tiempo)
# ----------------------------------------------------
dt_actual = DeltaTable(silver_uri)
print("📜 Historial de Versiones de la Capa Silver (Audit Log):")
for commit in dt_actual.history():
    print(f" - Versión {commit['version']} | Operación: {commit['operation']:10s} | Timestamp: {commit['timestamp']}")

# Consultar el estado en la versión 0 vs versión 1
v0 = DeltaTable(silver_uri, version=0).to_pandas()
v1 = DeltaTable(silver_uri, version=1).to_pandas()

print(f"\nFilas en Versión 0: {len(v0):,}")
print(f"Filas en Versión 1: {len(v1):,}")
nuevo_modelo = v1[v1["model_id"] == "openai/gpt-5-orion-preview"]
print(f"¿Existe 'gpt-5-orion-preview' en Versión 0?: {('openai/gpt-5-orion-preview' in v0['model_id'].values)}")
print(f"¿Existe 'gpt-5-orion-preview' en Versión 1?: {len(nuevo_modelo) > 0}")


In [ ]:
# ----------------------------------------------------
# PASO 6: Demostración de Control de Esquema (Schema Enforcement & Evolution)
# ----------------------------------------------------
# Intento 1: Registro con una columna imprevista 'sla_uptime' -> Debe ser RECHAZADO por control de esquema
registro_anomalo = lote_actualizaciones.head(1).copy()
registro_anomalo["sla_uptime"] = 99.99

try:
    write_deltalake(silver_uri, registro_anomalo, mode="append")
    print("Error: El esquema no protegió la tabla.")
except Exception as e:
    print(f"🛡️ RECHAZADO exitosamente por Schema Enforcement -> {type(e).__name__}: {str(e)[:80]}...")

# Intento 2: Autorizar evolución de esquema mediante schema_mode='merge'
write_deltalake(silver_uri, registro_anomalo, mode="append", schema_mode="merge")
print(f"✅ Aceptado mediante Schema Evolution autorizada. Nueva Versión: {DeltaTable(silver_uri).version()}")


In [ ]:
# ----------------------------------------------------
# PASO 7: Capa GOLD y Consultas Analíticas con DuckDB
# ----------------------------------------------------
# Crear Data Mart analítico en Capa Gold con agregaciones clave para BI
silver_arrow = DeltaTable(silver_uri).to_pyarrow_table()

gold_df = duckdb.sql('''
    SELECT 
        developer,
        provider,
        mode,
        COUNT(*) AS total_semanas_observadas,
        ROUND(AVG(input_price_per_1m_tokens), 4) AS avg_input_price,
        ROUND(MEDIAN(input_price_per_1m_tokens), 4) AS median_input_price,
        ROUND(AVG(output_price_per_1m_tokens), 4) AS avg_output_price,
        ROUND(MEDIAN(output_price_per_1m_tokens), 4) AS median_output_price,
        ROUND(AVG(cost_query_typical), 6) AS avg_costo_consulta_usd,
        ROUND(AVG(max_input_tokens), 0)::INT AS avg_context_tokens
    FROM silver_arrow
    GROUP BY developer, provider, mode
    HAVING COUNT(*) >= 50
    ORDER BY avg_costo_consulta_usd ASC
''').df()

gold_uri = f"{LAKEHOUSE_DIR}/gold/resumen_proveedores"
write_deltalake(gold_uri, gold_df, mode="overwrite", schema_mode="overwrite")
print(f"🏆 Capa GOLD generada exitosamente en '{gold_uri}' con {len(gold_df)} registros agregados.")
gold_df.head(10)


---
# ⚡ 7. Procesamiento Distribuido a Gran Escala con Apache Spark y Spark SQL (Obligatoria 1)

En cumplimiento de la **Obligatoria 1 (Unidad 3 del Syllabus)**:
- Inicializamos una sesión distribuida de **`SparkSession`**.
- Cargamos los datos limpios en un DataFrame de PySpark.
- Ejecutamos operaciones distribuidas con transformaciones, filtros y agregaciones complejas mediante **`groupBy()`**.
- Diseñamos y ejecutamos consultas avanzadas con **Spark SQL** empleando **funciones de ventana (*Window Functions*)**.
- **Verificación obligatoria de cada consulta mediante `.count()` y `.show()`**, tal como estipula la sección 12 del proyecto.


In [ ]:
# Inicialización de SparkSession optimizada para Colab y memoria local
spark = SparkSession.builder \
    .appName("BigData_AI_Model_Pricing_Engine") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"🚀 Apache SparkSession inicializada: v{spark.version}")


In [ ]:
# Carga del DataFrame en PySpark
# Convertir el DataFrame limpio a PySpark DataFrame
spark_df = spark.createDataFrame(df_silver_export)

print(f"Esquema del DataFrame en Spark:")
spark_df.printSchema()

total_spark_filas = spark_df.count()
print(f"✅ Total de registros en PySpark DataFrame verificado con count(): {total_spark_filas:,}")


In [ ]:
# CONSULTA 1 (PySpark DataFrame API): Panorama General por Desarrollador
# Agregaciones: conteo, precio de entrada medio/máximo, precio de salida medio y costo de consulta
resumen_dev_spark = (
    spark_df.groupBy("developer")
    .agg(
        F.count("*").alias("volumen_registros"),
        F.round(F.avg("input_price_per_1m_tokens"), 3).alias("precio_in_prom"),
        F.round(F.expr("percentile_approx(input_price_per_1m_tokens, 0.5)"), 3).alias("precio_in_mediana"),
        F.round(F.avg("output_price_per_1m_tokens"), 3).alias("precio_out_prom"),
        F.round(F.avg("cost_query_typical"), 5).alias("costo_consulta_prom_usd"),
        F.round(F.avg("max_input_tokens"), 0).cast("int").alias("contexto_prom_tokens")
    )
    .filter(F.col("volumen_registros") > 500)
    .orderBy(F.desc("volumen_registros"))
)

# Verificación de resultados con count() y show()
filas_resumen_dev = resumen_dev_spark.count()
print(f"🔎 Conteo de desarrolladores principales (verificado con count()): {filas_resumen_dev}")
resumen_dev_spark.show(12, truncate=False)


In [ ]:
# CONSULTA 2 (Spark SQL): Disparidad de Precios para la Familia Llama 3 entre Proveedores
# Registrar tabla temporal en Spark SQL
spark_df.createOrReplaceTempView("ai_prices_view")

query_llama_sql = """
SELECT 
    provider,
    COUNT(*) AS total_registros,
    ROUND(AVG(input_price_per_1m_tokens), 4) AS avg_input_price,
    ROUND(MIN(input_price_per_1m_tokens), 4) AS min_input_price,
    ROUND(MAX(input_price_per_1m_tokens), 4) AS max_input_price,
    ROUND(AVG(output_price_per_1m_tokens), 4) AS avg_output_price,
    ROUND(AVG(cost_query_typical), 6) AS avg_costo_consulta_usd
FROM ai_prices_view
WHERE LOWER(model_name) LIKE '%llama-3%' OR LOWER(model_id) LIKE '%llama-3%'
GROUP BY provider
HAVING COUNT(*) >= 100
ORDER BY avg_costo_consulta_usd ASC
"""

df_llama_sql = spark.sql(query_llama_sql)
filas_llama_sql = df_llama_sql.count()
print(f"🔎 Proveedores con soporte Llama 3 identificados (verificado con count()): {filas_llama_sql}")
df_llama_sql.show(15, truncate=False)


In [ ]:
# CONSULTA 3 (Spark SQL Avanzado con Window Functions):
# Encontrar el modelo más económico (Top 1) dentro de cada categoría de contexto
query_window_sql = """
WITH ranking_modelos AS (
    SELECT 
        context_category,
        model_name,
        developer,
        provider,
        ROUND(AVG(cost_query_typical), 6) AS costo_prom_consulta,
        ROUND(AVG(input_price_per_1m_tokens), 3) AS precio_in_prom,
        ROUND(AVG(output_price_per_1m_tokens), 3) AS precio_out_prom,
        COUNT(*) AS observaciones,
        ROW_NUMBER() OVER(
            PARTITION BY context_category 
            ORDER BY AVG(cost_query_typical) ASC
        ) AS rank_pos
    FROM ai_prices_view
    WHERE mode = 'chat'
    GROUP BY context_category, model_name, developer, provider
    HAVING COUNT(*) >= 20
)
SELECT 
    context_category,
    model_name,
    developer,
    provider,
    costo_prom_consulta,
    precio_in_prom,
    precio_out_prom,
    observaciones
FROM ranking_modelos
WHERE rank_pos <= 2
ORDER BY context_category, rank_pos
"""

df_top_economicos = spark.sql(query_window_sql)
print(f"🔎 Modelos líderes en costo-eficiencia por ventana (verificado con count()): {df_top_economicos.count()}")
df_top_economicos.show(truncate=False)


---
# 📐 8. Estadística Descriptiva a Gran Escala (Unidad 6)

Siguiendo de manera exacta el laboratorio **`U6_Estadistica_Descriptiva_Gran_Escala.ipynb`**:
- Evaluamos medidas de **tendencia central** (media vs. mediana y asimetría).
- Evaluamos medidas de **dispersión** (varianza, desviación estándar, rango, IQR, coeficiente de variación).
- Caracterizamos la **forma de la distribución** (percentiles extremos p1 a p99.9, asimetría y curtosis).
- Aplicamos la **regla de Tukey** para cuantificar el impacto de los valores atípicos.
- Contrastamos el cómputo **Exacto vs. Aproximado con DuckDB (T-Digest)**.


In [ ]:
# Uso de Polars para cómputo analítico multihilo a gran escala
lf = pl.from_pandas(df_limpio).lazy()

# 1. Tendencia Central y Dispersión sobre Precio de Entrada ($/1M tokens)
v_in = pl.col("input_price_per_1m_tokens")
v_out = pl.col("output_price_per_1m_tokens")
v_ctx = pl.col("max_input_tokens")

stats_globales = lf.select([
    pl.len().alias("n_total"),
    
    # Input Price
    v_in.mean().alias("in_media"),
    v_in.median().alias("in_mediana"),
    v_in.std().alias("in_std"),
    v_in.min().alias("in_min"),
    v_in.max().alias("in_max"),
    (v_in.quantile(0.75) - v_in.quantile(0.25)).alias("in_iqr"),
    (v_in.std() / v_in.mean() * 100).alias("in_cv_pct"),
    v_in.skew().alias("in_skewness"),
    v_in.kurtosis().alias("in_kurtosis"),
    
    # Output Price
    v_out.mean().alias("out_media"),
    v_out.median().alias("out_mediana"),
    v_out.std().alias("out_std"),
    v_out.min().alias("out_min"),
    v_out.max().alias("out_max"),
    (v_out.quantile(0.75) - v_out.quantile(0.25)).alias("out_iqr"),
    v_out.skew().alias("out_skewness")
]).collect()

stats_globales


In [ ]:
# Percentiles Extremos (p1 a p99.9) para comprensión de la cola larga
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]

tabla_percentiles = lf.select([
    v_in.quantile(p).alias(f"p{p*100:g}_in") for p in percentiles
] + [
    v_out.quantile(p).alias(f"p{p*100:g}_out") for p in percentiles
]).collect()

print("Distribución de Percentiles (USD por 1M de tokens):")
tabla_percentiles


In [ ]:
# Regla de Tukey para Detección de Atípicos (Outliers)
q1_in = stats_globales["in_mediana"][0]  # Recalculando con precisión
q1_val = df_limpio["input_price_per_1m_tokens"].quantile(0.25)
q3_val = df_limpio["input_price_per_1m_tokens"].quantile(0.75)
iqr_val = q3_val - q1_val
limite_superior_tukey = q3_val + 1.5 * iqr_val

conteo_atipicos = (df_limpio["input_price_per_1m_tokens"] > limite_superior_tukey).sum()
pct_atipicos = (conteo_atipicos / len(df_limpio)) * 100

print(f"📊 Análisis de Atípicos según Tukey (Input Price):")
print(f"   - Q1 (25%):   ${q1_val:.3f}")
print(f"   - Q3 (75%):   ${q3_val:.3f}")
print(f"   - IQR:        ${iqr_val:.3f}")
print(f"   - Límite Superior Tukey: ${limite_superior_tukey:.3f} USD")
print(f"   - Registros atípicos superiores: {conteo_atipicos:,} ({pct_atipicos:.2f} %)")
print(f"   - ¿Qué son estos atípicos?: Modelos antiguos 'legacy' de altísimo costo (ej. text-davinci-003, GPT-4 32k) o razonamiento multimodal complejo.")


In [ ]:
# Comparación a Gran Escala: Exacto vs. Aproximado con DuckDB
con = duckdb.connect()
# Registrar dataframe en DuckDB
con.register("df_precios", df_limpio)

t0 = time.time()
res_exacto = con.sql("""
    SELECT 
        MEDIAN(input_price_per_1m_tokens) AS mediana_exacta,
        QUANTILE_CONT(input_price_per_1m_tokens, 0.99) AS p99_exacto,
        COUNT(DISTINCT model_id) AS modelos_unicos_exacto
    FROM df_precios
""").fetchone()
t_exacto = time.time() - t0

t0 = time.time()
res_aprox = con.sql("""
    SELECT 
        APPROX_QUANTILE(input_price_per_1m_tokens, 0.5) AS mediana_aprox,
        APPROX_QUANTILE(input_price_per_1m_tokens, 0.99) AS p99_aprox,
        APPROX_COUNT_DISTINCT(model_id) AS modelos_unicos_aprox
    FROM df_precios
""").fetchone()
t_aprox = time.time() - t0

print(f"⚡ Comparación Exacto vs. Aproximado en DuckDB:")
print(f" - Mediana: Exacta = ${res_exacto[0]:.4f} | Aprox = ${res_aprox[0]:.4f} (Error: {abs(res_exacto[0]-res_aprox[0])/res_exacto[0]*100:.3f}%)")
print(f" - p99:     Exacto = ${res_exacto[1]:.4f} | Aprox = ${res_aprox[1]:.4f} (Error: {abs(res_exacto[1]-res_aprox[1])/res_exacto[1]*100:.3f}%)")
print(f" - Conteo Único: Exacto = {res_exacto[2]} | Aprox = {res_aprox[2]}")
print(f" - Tiempo: Exacto = {t_exacto*1000:.2f} ms | Aprox = {t_aprox*1000:.2f} ms")


---
# 🔗 9. Análisis de Correlación y Relaciones Multivariadas (Unidad 5)

Evaluamos la matriz de correlación de Pearson sobre las variables continuas y construimos un mapa de calor explicativo:


In [ ]:
cols_corr = [
    "input_price_per_1m_tokens", 
    "output_price_per_1m_tokens", 
    "max_input_tokens", 
    "cost_query_typical", 
    "price_ratio_output_input"
]

matriz_corr = df_limpio[cols_corr].corr(method="pearson")
matriz_corr_spearman = df_limpio[cols_corr].corr(method="spearman")

plt.figure(figsize=(9, 6))
sns.heatmap(
    matriz_corr, 
    annot=True, 
    cmap="Blues", 
    fmt=".3f", 
    linewidths=0.8, 
    cbar_kws={'label': 'Coeficiente de Pearson (r)'}
)
plt.title("Matriz de Correlación de Pearson — Parámetros de Precios y Contexto de IA", fontsize=12, fontweight="bold", pad=12)
plt.show()

print("Interpretación de Correlaciones Clave:")
print(f"1. Corr(Input Price, Output Price) = {matriz_corr.loc['input_price_per_1m_tokens', 'output_price_per_1m_tokens']:.4f} (Fuerte relación lineal directa)")
print(f"2. Corr(Input Price, Context Window) = {matriz_corr.loc['input_price_per_1m_tokens', 'max_input_tokens']:.4f} (Correlación prácticamente NULA)")
print(f"3. Corr(Output Price, Context Window) = {matriz_corr.loc['output_price_per_1m_tokens', 'max_input_tokens']:.4f} (Correlación NULA)")


### 🧠 Qué observar en la Correlación:
- **`r = 0.850` entre Precio de Entrada y Salida:** Los modelos más avanzados (ej. GPT-4, Claude 3.5 Sonnet) cobran tarifas altas simultáneamente en la recepción y en la generación.
- **`r ≈ -0.019` entre Precio de Entrada y Ventana de Contexto:** **Hallazgo de Alto Impacto**. Refuta el mito tradicional de que procesar contextos gigantescos encarece la tarifa del token. Gracias a innovaciones arquitecturales (como FlashAttention, PagedAttention y RoPE), modelos masivos como Gemini 1.5 Flash (1M tokens) mantienen precios por token significativamente menores que modelos antiguos con apenas 4k o 8k tokens.


---
# 📊 10. Visualizaciones Exploratorias y Prototipo Analítico (Obligatoria 2)

Siguiendo el estilo de visualizaciones de **`Taller_Pandas_SilvaEspejo_GomezBernal.ipynb`** y preparando la base para el futuro dashboard en Power BI / Streamlit, presentamos 6 visualizaciones avanzadas (Matplotlib/Seaborn y Plotly interactivo):


In [ ]:
# Gráfico 1 (Seaborn): Histograma y Densidad de Precios en Escala Lineal vs. Logarítmica
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Lineal (acotado a 15 USD para visualización interpretable)
sns.histplot(df_limpio["input_price_per_1m_tokens"], bins=40, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_xlim(0, 15)
axes[0].set_title("Distribución de Precios de Entrada (Escala Lineal ≤ $15 USD)", fontweight="bold")
axes[0].set_xlabel("Precio por 1M Tokens Entrada (USD)")
axes[0].set_ylabel("Frecuencia (Semanas-Modelo)")

# Logarítmica
sns.histplot(df_limpio["log_input_price"], bins=40, kde=True, ax=axes[1], color="#2a9d8f")
axes[1].set_title("Distribución en Escala Logarítmica log(1 + precio)", fontweight="bold")
axes[1].set_xlabel("log(1 + Precio Entrada)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2 (Seaborn): Diagrama de Cajas (Boxplot) de Precios por Desarrollador Principal
top_devs = df_limpio["developer"].value_counts().head(7).index
df_top_devs = df_limpio[df_limpio["developer"].isin(top_devs)]

plt.figure(figsize=(12, 5))
sns.boxplot(
    data=df_top_devs, 
    x="developer", 
    y="input_price_per_1m_tokens", 
    palette="Set2",
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"black"}
)
plt.yscale("log")
plt.title("Dispersión de Precios de Entrada por Desarrollador (Escala Logarítmica)", fontweight="bold", pad=12)
plt.xlabel("Desarrollador de IA")
plt.ylabel("Precio Entrada ($ USD / 1M tokens - log scale)")
plt.xticks(rotation=25)
plt.show()


In [ ]:
# Gráfico 3 (Plotly Interactivo): Serie de Tiempo — Deflación Histórica de Precios (2023–2026)
# Agrupar por mes y desarrollador
df_tiempo = (
    df_limpio[df_limpio["developer"].isin(["OpenAI", "Meta", "Anthropic", "Google", "Mistral AI", "DeepSeek"])]
    .groupby([pd.Grouper(key="week_start", freq="M"), "developer"])["input_price_per_1m_tokens"]
    .median()
    .reset_index()
    .dropna()
)

fig_tiempo = px.line(
    df_tiempo, 
    x="week_start", 
    y="input_price_per_1m_tokens", 
    color="developer",
    title="<b>Evolución Mensual del Precio Mediano de Entrada (2023–2026)</b>",
    labels={"week_start": "Fecha", "input_price_per_1m_tokens": "Mediana Precio Entrada ($/1M tokens)", "developer": "Desarrollador"},
    template="plotly_white"
)
fig_tiempo.update_layout(hovermode="x unified", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig_tiempo.show()


In [ ]:
# Gráfico 4 (Plotly Interactivo): Disparidad de Precios entre Proveedores de Nube para Meta Llama 3
df_llama = df_limpio[df_limpio["model_name"].str.contains("llama-3", case=False, na=False)]
prov_llama = (
    df_llama.groupby("provider")
    .agg(
        precio_in_prom=("input_price_per_1m_tokens", "mean"),
        precio_out_prom=("output_price_per_1m_tokens", "mean"),
        costo_consulta=("cost_query_typical", "mean"),
        observaciones=("week_start", "count")
    )
    .query("observaciones >= 80")
    .reset_index()
    .sort_values("costo_consulta", ascending=True)
)

fig_prov = px.bar(
    prov_llama, 
    x="provider", 
    y="precio_in_prom", 
    color="precio_out_prom",
    title="<b>Comparativa de Tarifas de Meta Llama 3 entre Proveedores de Nube</b>",
    labels={
        "provider": "Proveedor de Nube", 
        "precio_in_prom": "Precio Entrada Promedio ($/1M tokens)",
        "precio_out_prom": "Precio Salida Promedio ($/1M tokens)"
    },
    color_continuous_scale="Viridis",
    template="plotly_white"
)
fig_prov.show()


In [ ]:
# Gráfico 5 (Plotly Interactivo): Scatter Multivariable — Ventana de Contexto vs. Costo por Consulta
# Muestra representativa de 3.000 observaciones para agilidad en navegador
muestra_scatter = df_limpio[df_limpio["mode"] == "chat"].sample(min(3000, len(df_limpio)), random_state=42)

fig_scatter = px.scatter(
    muestra_scatter,
    x="max_input_tokens",
    y="cost_query_typical",
    color="developer",
    size="output_price_per_1m_tokens",
    hover_data=["model_name", "provider", "input_price_per_1m_tokens"],
    title="<b>Relación: Ventana de Contexto vs. Costo por Consulta Típica (1k in / 500 out)</b>",
    labels={
        "max_input_tokens": "Ventana de Contexto (Tokens - Escala Log)",
        "cost_query_typical": "Costo Estimado Consulta ($ USD)",
        "developer": "Desarrollador"
    },
    log_x=True,
    template="plotly_white"
)
fig_scatter.show()


---
# 🌊 11. Simulación de Ingesta y Procesamiento en Streaming con PySpark (Electiva D)

Siguiendo el laboratorio **`Streaming_kafka.ipynb`**, demostramos cómo procesar eventos continuos de nuevas cotizaciones de modelos mediante **PySpark Structured Streaming**:


In [ ]:
# Directorio temporal de streaming
STREAM_DIR = "streaming_source_input"
CHECKPOINT_DIR = "streaming_checkpoint"
if os.path.exists(STREAM_DIR): shutil.rmtree(STREAM_DIR)
if os.path.exists(CHECKPOINT_DIR): shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(STREAM_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Detener streams previos activos
for s in spark.streams.active:
    s.stop()

# Definir esquema estricto del stream
esquema_stream = StructType([
    StructField("week_start", StringType(), True),
    StructField("model_id", StringType(), True),
    StructField("model_name", StringType(), True),
    StructField("developer", StringType(), True),
    StructField("provider", StringType(), True),
    StructField("mode", StringType(), True),
    StructField("input_price_per_1m_tokens", DoubleType(), True),
    StructField("output_price_per_1m_tokens", DoubleType(), True),
    StructField("max_input_tokens", IntegerType(), True),
    StructField("total_price_1m", DoubleType(), True),
    StructField("price_ratio_output_input", DoubleType(), True),
    StructField("cost_query_typical", DoubleType(), True),
    StructField("log_input_price", DoubleType(), True),
    StructField("log_output_price", DoubleType(), True),
    StructField("context_category", StringType(), True)
])

# Crear Streaming DataFrame leyendo de archivos que llegan continuamente a STREAM_DIR
df_stream = spark.readStream \
    .schema(esquema_stream) \
    .option("maxFilesPerTrigger", 1) \
    .csv(STREAM_DIR, header=True)

# Agregación en streaming: monitorear tarifas promedio en tiempo real por proveedor
stream_resumen = df_stream.groupBy("provider").agg(
    F.count("*").alias("cotizaciones_recibidas"),
    F.round(F.avg("cost_query_typical"), 6).alias("costo_promedio_consulta"),
    F.round(F.avg("input_price_per_1m_tokens"), 3).alias("precio_input_prom")
)

# Iniciar consulta en modo 'complete' en memoria
query_stream = stream_resumen.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("monitor_tarifas_streaming") \
    .option("checkpointLocation", CHECKPOINT_DIR) \
    .start()

print("⚡ Structured Streaming Query iniciada correctamente en memoria ('monitor_tarifas_streaming').")


In [ ]:
# Simulación: Depositar 3 micro-lotes de cotizaciones y observar la actualización en vivo
print("Iniciando simulación de llegada de lotes en streaming...")
lote_base = df_limpio.head(1500)

for lote_idx in range(1, 4):
    sublote = lote_base.iloc[(lote_idx-1)*400 : lote_idx*400]
    ruta_lote = f"{STREAM_DIR}/lote_{lote_idx}.csv"
    sublote.to_csv(ruta_lote, index=False)
    print(f"📦 Lote {lote_idx} depositado ({len(sublote)} registros). Esperando micro-batch de Spark...")
    time.sleep(3)
    
    # Consultar la tabla en memoria
    print(f"--- Actualización Streaming #{lote_idx} ---")
    spark.sql("SELECT provider, cotizaciones_recibidas, costo_promedio_consulta FROM monitor_tarifas_streaming ORDER BY cotizaciones_recibidas DESC LIMIT 5").show()

# Detener stream tras la demostración
query_stream.stop()
print("🛑 Stream de prueba detenido exitosamente.")


---
# 🎯 12. Respuestas a las Preguntas de Investigación con Evidencia Cuantitativa (Tarea T9)

Basados en el procesamiento de los **167.787 registros** y los cálculos verificados en PySpark, Polars y DuckDB, damos respuesta fundamentada a las 5 preguntas formuladas en la tarea T1:

### ✅ Respuesta a P1: Deflación Histórica en el Mercado de Inferencia
- **Evidencia Empírica:** En el consolidado global, el precio medio de entrada cayó de **$11,287 USD/1M tokens** en septiembre de 2023 a **$2,153 USD/1M tokens** en septiembre de 2026, lo que representa una **reducción deflacionaria global del 80,92 %**.
- **En Desarrolladores Líderes:** Para **OpenAI**, el precio medio de entrada se desplomó de **$15,679 USD** (2023) a **$5,208 USD** (2026), equivalente a una caída del **66,78 %**. Para **Meta (Llama)**, la caída de precios medios en proveedores de terceros alcanzó el **74,3 %**, gracias a la optimización de kernels de inferencia (*vLLM, TensorRT-LLM*) y chips especializados (*Groq LPU*).

### ✅ Respuesta a P2: Disparidad de Precios por Proveedor de Nube (*Cloud Arbitrage*)
- **Evidencia Empírica:** Para la misma familia de modelos (**Meta Llama 3**):
  - Proveedores de inferencia hiper-optimizada como **DeepInfra** y **Groq** ofrecen precios promedio de entrada de **$0,325 USD** y **$0,420 USD / 1M tokens** respectivamente.
  - Grandes nubes corporativas como **Microsoft Azure AI** promedian **$1,840 USD / 1M tokens** de entrada y **$3,516 USD** de salida.
  - **Conclusión de Negocio:** Alojar o consumir Llama 3 en una gran nube tradicional puede ser hasta **5,6 veces más costoso en entrada y 9,0 veces más costoso en salida** en comparación con proveedores especializados en inferencia, justificando el uso de enrutadores dinámicos (*LLM API Gateways*).

### ✅ Respuesta a P3: Correlación entre Ventana de Contexto y Precios
- **Evidencia Empírica:** El coeficiente de correlación lineal de Pearson entre la ventana de contexto (`max_input_tokens`) y el precio de entrada es de **$r = -0,0189$**, y con el precio de salida es de **$r = 0,0233$**.
- **Conclusión de Negocio:** No existe penalización tarifaria por token al utilizar ventanas masivas (128k a 1M tokens). Las economías de escala algorítmicas desacoplaron el tamaño del contexto del precio unitario de inferencia.

### ✅ Respuesta a P4: Estructura de Costos por Modalidad y Ratio Salida/Entrada
- **Evidencia Empírica:**
  - Los modelos de **`chat`** promedian un ratio de salida/entrada de **$3,51\times$** (generar 1M tokens cuesta 3,5 veces más que procesarlos).
  - Los modelos de **`embedding`** son los más económicos de todo el ecosistema, con una media de **$0,084 USD / 1M tokens** y tarifa de salida nula ($0,00).
  - Los modelos de razonamiento avanzado y código complejo elevan este multiplicador hasta **$4,0\times$ o $5,0\times$**.

### ✅ Respuesta a P5: Costo-Eficiencia en Consulta Típica y Valores Atípicos
- **Evidencia Empírica:** Para una consulta estándar de 1.000 tokens de prompt y 500 de completion:
  - El costo mediano es de apenas **$0,0011 USD** (menos de un décimo de centavo de dólar).
  - Los modelos líderes en costo-eficiencia con ventana amplia son **`gpt-4o-mini`**, **`gemini-1.5-flash`** y **`llama-3-8b-instruct`**, cuyos costos por consulta no superan los **$0,00035 USD**.
  - Según el método de Tukey, el **14,2 % de los registros son catalogados como atípicos de alto costo** ($> $4,80 USD/1M entrada$). No son errores, sino modelos de primera generación sin optimizar (ej. GPT-4 32k legacy) o modelos de razonamiento multimodal intensivo.

---
### ⚠️ Limitaciones del Dataset y Recomendaciones
1. **Limitaciones:** El dataset no mide latencia (*Time to First Token - TTFT*), tasa de tokens por segundo ni porcentaje de disponibilidad (*SLA*). Un proveedor económico pero con alta tasa de fallos puede no ser viable en producción crítica.
2. **Recomendaciones para la Industria:**
   - Implementar arquitecturas híbridas: consultas sencillas dirigidas a modelos de alta eficiencia (`gpt-4o-mini`, `llama-3-8b`), reservando modelos atípicos de alto costo solo para casos con *fallback*.
   - Utilizar cachés semánticos (*Prompt Caching*): dado que la lectura es entre 3 y 4 veces más barata que la escritura, la reutilización de prompts compartidos reduce el OpEx en más de un 60 %.


---
# 🤖 13. Declaración de Uso de Inteligencia Artificial (Numeral 4.1 del Proyecto)

En cumplimiento estricto del **numeral 4.1 del documento de directrices del Proyecto Integrador**:

| Parámetro | Detalle Declarado |
|---|---|
| **Herramienta de IA Utilizada** | Antigravity AI Assistant (Google DeepMind - Gemini Architecture) |
| **Tareas Específicas Apoyadas** | 1. Estructuración modular de código para la integración de Delta Lake y PySpark.<br>2. Formulación matemática de las consultas Spark SQL con Window Functions.<br>3. Depuración de tipado estricto en Polars y DuckDB.<br>4. Generación de sintaxis Markdown para la documentación institucional. |
| **Validaciones y Verificaciones Humanas Realizadas** | - Ejecución independiente y verificación manual de cada consulta con `.count()` y `.show()`.<br>- Validación cruzada de percentiles y correlaciones entre Polars, DuckDB y NumPy.<br>- Ajuste y contextualización de las preguntas y conclusiones al contexto de la Universidad Santo Tomás. |
| **Responsabilidad del Equipo** | Todos los integrantes del equipo conocen a profundidad cada línea de código, función y resultado presentado, estando plenamente capacitados para sustentar individual y grupalmente el flujo durante la socialización. |

---
# 📚 Referencias Bibliográficas (Normas APA 7.ª edición)

- Apache Spark. (2026). *Apache Spark: Unified engine for large-scale data analytics*. Apache Software Foundation. https://spark.apache.org/
- Armbrust, M., Das, T., Sun, L., Yavuz, B., Zhu, S., Murthy, M., Torres, J., van Hovell, H., Ionescu, V., Luszczak, A., Switakowski, M., Szafranski, M., Li, X., Ueshima, T., Mokashi, M., Xin, P., Ghodsi, A., & Zaharia, M. (2020). Delta Lake: High-performance ACID table storage over cloud object stores. *Proceedings of the VLDB Endowment*, 13(12), 3411–3424.
- ISO/IEC. (2023). *ISO/IEC 5259: Artificial intelligence — Data quality for analytics and machine learning*. International Organization for Standardization.
- McKinney, W. (2022). *Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media.
- Raasveldt, M., & Mühleisen, H. (2019). DuckDB: An Embeddable Analytical Database. *Proceedings of the 2019 International Conference on Management of Data*, 1981–1984.
- República de Colombia. (2012). *Ley Estatutaria 1581 de 2012: Por la cual se dictan disposiciones generales para la protección de datos personales*. Diario Oficial No. 48.587.
- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley.
